# Tariff example usage
This notebook demonstrates how to use the `Tariff` class to calculate energy costs based on a given tariff structure. We will create a sample tariff and then use it to compute the cost of energy consumption over a specified period.

### 1. Creating an index
Most tariffs are based on one or more indexes. For example, a common structure is to have a fixed cost component and a variable cost component that depends on the day-ahead electricity price. Let's create an index for the day-ahead price using the `EntsoeDayAheadIndex`:

In [1]:
from os import environ

from energy_cost.index import EntsoeDayAheadIndex, Index

Index.register("Belpex15min", EntsoeDayAheadIndex(country_code="BE", api_key=environ["ENTSOE_API_KEY"]))

### 2. Defining a tariff
The easiest way to define a tariff is to specify it in a YAML file. A tariff is a list of versions, and each version becomes active from its `start` timestamp.

We'll start with the simplest form: a version with a `consumption` formula.

A simple dynamic tariff might look like this:

In [2]:
from helpers import display_yaml  # ty: ignore[unresolved-import]

display_yaml("../data/tariffs/dynamic.yml")

```yaml
- start: 2025-01-01T00:00:00+01:00 # the start of validity of this tariff version
  consumption: # the cost formula for consumption
    constant_cost: 10.0 # the constant cost component of the price formula in €/MWh
    variable_costs: # the variable cost components of the price formula in €/MWh
    - index: Belpex15min # the name of the index to use for this variable cost component
      scalar: 1.05 # the scalar to apply to the index values for this variable cost component
- start: 2026-01-01T00:00:00+01:00 # from this date, a new tariff version applies with a different price formula
  consumption:
    constant_cost: 12.0
    variable_costs:
    - index: Belpex15min
      scalar: 1.10
```

In [3]:
from energy_cost.tariff import Tariff

tariff = Tariff.from_yaml("../data/tariffs/dynamic.yml")
tariff

Tariff(versions=[TariffVersion(start=datetime.datetime(2025, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), injection={}, consumption={'all': {<CostType.ENERGY: 'energy'>: IndexFormula(kind='index', constant_cost=10.0, variable_costs=[IndexAdder(index='Belpex15min', scalar=1.05)])}}, capacity=None, periodic={}), TariffVersion(start=datetime.datetime(2026, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), injection={}, consumption={'all': {<CostType.ENERGY: 'energy'>: IndexFormula(kind='index', constant_cost=12.0, variable_costs=[IndexAdder(index='Belpex15min', scalar=1.1)])}}, capacity=None, periodic={})])

### 3. Calculating cost per energy consumption in a given period
Once we have defined our tariff, we can use it to calculate the cost of energy consumption over a specified period. This is done by calling the `get_cost` method of the `Tariff` class, which returns a DataFrame with the cost for each time step in the given period.

In [4]:
import datetime as dt

start = dt.datetime.fromisoformat("2026-03-08 00:00:00+02:00")
end = dt.datetime.fromisoformat("2026-03-10 00:00:00+02:00")
tariff.get_cost(start, end)

,timestamp,energy,total
0,2026-03-08 00:00:00+02:00,156.111,156.111
1,2026-03-08 00:15:00+02:00,143.516,143.516
2,2026-03-08 00:30:00+02:00,140.458,140.458
3,2026-03-08 00:45:00+02:00,135.057,135.057
4,2026-03-08 01:00:00+02:00,162.392,162.392
...,...,...,...
187,2026-03-09 22:45:00+02:00,161.061,161.061
188,2026-03-09 23:00:00+02:00,182.808,182.808
189,2026-03-09 23:15:00+02:00,172.083,172.083
190,2026-03-09 23:30:00+02:00,171.731,171.731


### 4. Specifying injection tariffs
The `Tariff` class also supports injection tariffs, which can be used to calculate the revenue from feeding energy back into the grid. To do this, add an `injection` formula next to `consumption` in the tariff version, then call `get_cost` with `direction=PowerDirection.INJECTION`.

In [5]:
display_yaml("../data/tariffs/injection.yml")

```yaml
# Separate injection and consumption formulas
- start: 2025-01-01T00:00:00+01:00
  injection:
    constant_cost: -14.56
    variable_costs:
    - index: Belpex15min
      scalar: 0.9876
  consumption:
    constant_cost: 12.34
    variable_costs:
    - index: Belpex15min
      scalar: 1.23
```

In [6]:
from energy_cost.tariff import PowerDirection

tariff = Tariff.from_yaml("../data/tariffs/injection.yml")
tariff.get_cost(start, end, direction=PowerDirection.INJECTION)

,timestamp,energy,total
0,2026-03-08 00:00:00+02:00,114.825476,114.825476
1,2026-03-08 00:15:00+02:00,103.517456,103.517456
2,2026-03-08 00:30:00+02:00,100.771928,100.771928
3,2026-03-08 00:45:00+02:00,95.922812,95.922812
4,2026-03-08 01:00:00+02:00,120.464672,120.464672
...,...,...,...
187,2026-03-09 22:45:00+02:00,119.269676,119.269676
188,2026-03-09 23:00:00+02:00,138.794528,138.794528
189,2026-03-09 23:15:00+02:00,129.165428,129.165428
190,2026-03-09 23:30:00+02:00,128.849396,128.849396


### 5. Specifying meter types
The `Tariff` class also supports different prices per meter type, such as `single_rate`, `tou_peak`, and `tou_offpeak`. When a tariff depends on meter type, you can place those formulas under `consumption` or `injection`.

If needed, you can also add an `all` entry for formulas that apply to every meter type, and override them with a more specific meter type.

In [7]:
display_yaml("../data/tariffs/metered.yml")

```yaml
# Injection shared across all meter types; consumption varies by meter type
- start: 2025-01-01T00:00:00+01:00
  injection:
    constant_cost: -14.56
    variable_costs:
    - index: Belpex15min
      scalar: 0.9876
  consumption:
    tou_peak:
      constant_cost: 23.45
      variable_costs:
      - index: Belpex15min
        scalar: 2.34
    tou_offpeak:
      constant_cost: 10.0
      variable_costs:
      - index: Belpex15min
        scalar: 0.56

```

In [8]:
from energy_cost.tariff import MeterType

tariff = Tariff.from_yaml("../data/tariffs/metered.yml")
tariff.get_cost(start, end, meter_type=MeterType.TOU_PEAK)

,timestamp,energy,total
0,2026-03-08 00:00:00+02:00,330.0134,330.0134
1,2026-03-08 00:15:00+02:00,303.2204,303.2204
2,2026-03-08 00:30:00+02:00,296.7152,296.7152
3,2026-03-08 00:45:00+02:00,285.2258,285.2258
4,2026-03-08 01:00:00+02:00,343.3748,343.3748
...,...,...,...
187,2026-03-09 22:45:00+02:00,340.5434,340.5434
188,2026-03-09 23:00:00+02:00,386.8052,386.8052
189,2026-03-09 23:15:00+02:00,363.9902,363.9902
190,2026-03-09 23:30:00+02:00,363.2414,363.2414


### 6. Complex cost components
Some tariffs split the consumption price into separate components. In the example below, `consumption` includes `energy`, `renewable_certificates`, and `chp_certificates`.

This allows `get_cost` to return one column per component, together with the combined `total`.

In [9]:
display_yaml("../data/tariffs/cost_types.yml")

```yaml
# Multiple cost types in consumption
- start: 2025-01-01T00:00:00+01:00
  injection:
    constant_cost: -14.56
    variable_costs:
    - index: Belpex15min
      scalar: 0.9876
  consumption:
    renewable_certificates:
      constant_cost: 12.0
    chp_certificates:
      constant_cost: 4.56
    energy:
      constant_cost: 12.34
      variable_costs:
      - index: Belpex15min
        scalar: 1.23
```

In [10]:
tariff = Tariff.from_yaml("../data/tariffs/cost_types.yml")
tariff.get_cost(start, end)

,timestamp,renewable_certificates,chp_certificates,energy,total
0,2026-03-08 00:00:00+02:00,12.0,4.56,173.4823,190.0423
1,2026-03-08 00:15:00+02:00,12.0,4.56,159.3988,175.9588
2,2026-03-08 00:30:00+02:00,12.0,4.56,155.9794,172.5394
3,2026-03-08 00:45:00+02:00,12.0,4.56,149.9401,166.5001
4,2026-03-08 01:00:00+02:00,12.0,4.56,180.5056,197.0656
...,...,...,...,...,...
187,2026-03-09 22:45:00+02:00,12.0,4.56,179.0173,195.5773
188,2026-03-09 23:00:00+02:00,12.0,4.56,203.3344,219.8944
189,2026-03-09 23:15:00+02:00,12.0,4.56,191.3419,207.9019
190,2026-03-09 23:30:00+02:00,12.0,4.56,190.9483,207.5083


### 7. Periodic costs
Some tariffs also include fixed costs that do not depend on energy usage, such as a daily connection fee or a monthly service fee.

These are defined under `periodic`. Use `get_periodic_cost` to calculate the prorated cost over a given time interval:


In [11]:
display_yaml("../data/tariffs/periodic.yml")

```yaml
# Fixed periodic costs
- start: 2025-01-01T00:00:00+01:00
  periodic:
    connection_fee:
      period: daily
      constant_cost: 1.5
    service_fee:
      period: monthly
      constant_cost: 31.0

```

In [12]:
periodic_tariff = Tariff.from_yaml("../data/tariffs/periodic.yml")
periodic_tariff.get_periodic_cost(start, end)

{'connection_fee': 3.0, 'service_fee': 2.0}

### 8. Scheduled (time-of-use) price formulas

Some tariffs apply different prices depending on the **day of the week** and/or **time of day**, for example a weekday morning peak and a cheaper fallback during the remaining hours.

For this kind of tariff, `consumption` is written as a list of formulas with optional `when` conditions. The formulas are checked in order, the first match wins, and a formula without `when` acts as a fallback:


In [13]:
display_yaml("../data/tariffs/scheduled.yml")

```yaml
- start: 2025-01-01T00:00:00+01:00
  consumption:
    schedule:
      - when: # formula valid for weekday mornings and weekend daytime
          - days: [monday, tuesday, wednesday, thursday, friday]
            start: "06:00:00"
            end: "10:00:00"
          - days: [saturday, sunday]
            start: "07:00:00"
            end: "19:00:00"
        formula:
          constant_cost: 300.0
      - when: # formula valid any day during late morning and evening (the first matching formula takes precedence, so this will not apply during the weekday morning and weekend daytime windows above)
          - start: "10:00:00"
            end: "13:00:00"
          - start: "18:00:00"
            end: "22:00:00"
        formula:
          constant_cost: 150.0
      - formula:
          constant_cost: 100.0 # formula valid at all times not covered by the previous definitions

```

In [14]:
import datetime as dt

scheduled_tariff = Tariff.from_yaml("../data/tariffs/scheduled.yml")

# One full weekday at hourly resolution — Monday 2026-03-16
weekday_start = dt.datetime.fromisoformat("2026-03-16 00:00:00+01:00")
weekday_end = dt.datetime.fromisoformat("2026-03-17 00:00:00+01:00")

scheduled_tariff.get_cost(weekday_start, weekday_end, resolution=dt.timedelta(hours=1))

,timestamp,energy,total
0,2026-03-16 00:00:00+01:00,100.0,100.0
1,2026-03-16 01:00:00+01:00,100.0,100.0
2,2026-03-16 02:00:00+01:00,100.0,100.0
3,2026-03-16 03:00:00+01:00,100.0,100.0
4,2026-03-16 04:00:00+01:00,100.0,100.0
5,2026-03-16 05:00:00+01:00,100.0,100.0
6,2026-03-16 06:00:00+01:00,300.0,300.0
7,2026-03-16 07:00:00+01:00,300.0,300.0
8,2026-03-16 08:00:00+01:00,300.0,300.0
9,2026-03-16 09:00:00+01:00,300.0,300.0


And the same tariff over a full weekend day:


In [15]:
# One full weekend day at hourly resolution — Saturday 2026-03-21
weekend_start = dt.datetime.fromisoformat("2026-03-21 00:00:00+01:00")
weekend_end = dt.datetime.fromisoformat("2026-03-22 00:00:00+01:00")

scheduled_tariff.get_cost(weekend_start, weekend_end, resolution=dt.timedelta(hours=1))

,timestamp,energy,total
0,2026-03-21 00:00:00+01:00,100.0,100.0
1,2026-03-21 01:00:00+01:00,100.0,100.0
2,2026-03-21 02:00:00+01:00,100.0,100.0
3,2026-03-21 03:00:00+01:00,100.0,100.0
4,2026-03-21 04:00:00+01:00,100.0,100.0
5,2026-03-21 05:00:00+01:00,100.0,100.0
6,2026-03-21 06:00:00+01:00,100.0,100.0
7,2026-03-21 07:00:00+01:00,300.0,300.0
8,2026-03-21 08:00:00+01:00,300.0,300.0
9,2026-03-21 09:00:00+01:00,300.0,300.0


## 9. Capacity-based tariffs
Some tariffs are not based on consumption but on the maximum power capacity used during a period.
To define a capacity-based tariff we need to define two periods:
- `measurement_period`: the period over which the capacity is measured, capacity is defined as the sum of consumption values during this period
- `billing_period`: the period over which the maximum capacity is calculated and billed, this can be the same as the measurement period or a multiple of it

The actual prices are defined using the same formulas as for consumption-based tariffs, but placed under `capacity` instead of `consumption`. In this case all values are assumed to be in €/MW instead of €/MWh, and the cost is calculated as `capacity * price` instead of `consumption * price`.

For advanced use cases, we also allow defing `window_periods`, which defines a rolling window of `x` billing periods that are avaraged out to calculate the capacities. This is used in eg the Flemish capacity tariff, where the last 12 months are smoothed out to calculate the capacity for the next billing period.

In [16]:
display_yaml("../data/tariffs/capacity.yml")

```yaml
- start: 2025-01-01T00:00:00+01:00
  capacity: # define the cost formula for capacity based costs
    measurement_period: PT15M # the period over which capacity is measured in ISO 8601 duration format
    billing_period: P1M # the period over which capacity costs are billed in ISO 8601 duration format
    formula: # any formula definition supported for consumption costs can be used here, values are interpreted in €/MW instead of €/MWh
      constant_cost: 5.0 # eg a constant cost component of 5 €/MW
```

For capacity-based tariffs, we don't provide a multiplier frame (so these costs won't be indluded in `get_cost`), but instead you can call `apply_capacity_cost` which takes the consumption data as input and calculates the capacity costs based on that.

In [23]:
import pandas as pd

data = pd.DataFrame({
    "timestamp": pd.date_range(start="2026-01-01", end="2026-03-31", freq="1min", tz="UTC"),
    "value": 0.1,  # 100 kW constant consumption every 1 minute or 1.5 MW per 15 minutes
})

capacity_tariff = Tariff.from_yaml("../data/tariffs/capacity.yml")
capacity_tariff.apply_capacity_cost(data)

,timestamp,value
0,2026-01-01 00:00:00+00:00,7.5
1,2026-02-01 00:00:00+00:00,7.5
2,2026-03-01 00:00:00+00:00,7.5


## 10. Tiered formulas
Some capacity tariffs have tiered prices, where the price per MW changes depending on the total capacity used. This can be defined using the `bands` which specify the price for different capacity ranges. For example, usage below a 0.5 MW might be charged at a fixed monthly price, while usage above 0.5 MW is charged at a by usage.


In [ ]:
display_yaml("../data/tariffs/tiered.yml")

```yaml
- start: 2025-01-01T00:00:00+01:00
  capacity:
    measurement_period: PT15M 
    billing_period: PT1M
    window_periods: 12 # Setting this means the capacity will be avaraged over the last 12 billing periods (eg 12 months if billing_period is P1M) instead of being based on the single billing period in which the cost is being calculated, as is used by the Flemish capacity tariff.
    formula: 
      bands:
      - up_to: 0.5 # if capacity in a billing period does not exceed 0.5 MW, it is billed according to this formula
        formula: # a fixed cost component of 5 €/month
          period: monthly
          constant_cost: 5.0 
      - formula: # if capacity in a billing period exceeds 0.5 MW, it is billed according to this formula
          constant_cost: 10.0 # usage based costs of 10 €/MW
```

When calculating capacity cost, we can also already provide the billing period capacities, instead of the measurement period consumption data. In this case, the `apply_capacity_cost` method will skip the steps of resampling and calculating the maximum capacity per billing period, and directly apply the tiered prices to the provided capacities.

In [39]:
import pandas as pd

data = pd.DataFrame({
    "timestamp": pd.date_range(start="2025-01-01", end="2026-03-31", freq="1MS", tz="UTC"),
    # the low values in the first two months are vissible in the next 12 months due to the window_periods setting in the tiered tariff
    "value": [0.1, 0.1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
})

capacity_tariff = Tariff.from_yaml("../data/tariffs/tiered.yml")
capacity_tariff.apply_capacity_cost(data)

,timestamp,value
0,2025-01-01 00:00:00+00:00,5.000000
1,2025-02-01 00:00:00+00:00,5.000000
2,2025-03-01 00:00:00+00:00,5.000000
3,2025-04-01 00:00:00+00:00,5.500000
4,2025-05-01 00:00:00+00:00,6.400000
5,2025-06-01 00:00:00+00:00,7.000000
6,2025-07-01 00:00:00+00:00,7.428571
7,2025-08-01 00:00:00+00:00,7.750000
8,2025-09-01 00:00:00+00:00,8.000000
9,2025-10-01 00:00:00+00:00,8.200000
